# Flatten nested JSON in Python with chDB (drop-in pandas)

Companion to [flatten-nested-json-python](https://clickhouse.com/resources/engineering/flatten-nested-json-python).

Run `./generate.sh` first to create `data/`. Requires `pip install chdb pandas`.

## 1. Read a nested JSON file into a DataFrame

In [ ]:
import chdb.datastore as pd

df = pd.read_json("data/orders.json")
df

In [ ]:
df.dtypes

## 2. Inspect the nested columns

In [ ]:
pdf0 = df.to_pandas()
print("customer[0]:", pdf0["customer"].iloc[0])
print("items[0]:   ", pdf0["items"].iloc[0])

## 3. Explode items: one row per line item

In [ ]:
flat = df.explode("items")
flat[["order_id", "items"]]

## 4. Extract nested fields + compute line_total

In [ ]:
flat_pdf = flat.to_pandas().dropna(subset=["items"])
flat_pdf["country"] = flat_pdf["customer"].apply(lambda c: c["country"])
flat_pdf["tier"]    = flat_pdf["customer"].apply(lambda c: c["tier"])
flat_pdf["sku"]     = flat_pdf["items"].apply(lambda i: i["sku"])
flat_pdf["qty"]     = flat_pdf["items"].apply(lambda i: i["qty"])
flat_pdf["price"]   = flat_pdf["items"].apply(lambda i: i["price"])
flat_pdf["line_total"] = (flat_pdf["qty"] * flat_pdf["price"]).round(2)
flat_pdf[["order_id", "country", "tier", "sku", "qty", "line_total"]].sort_values(["order_id", "sku"])

## 5. Aggregate: revenue per country

In [ ]:
flat_pdf.groupby("country")["line_total"].sum().round(2).sort_values(ascending=False)

## 6. Hand off to real pandas when you need it

In [ ]:
pdf = df.to_pandas()   # a real pandas.DataFrame, in memory
type(pdf)

## 7. Performance: chDB read_json + explode vs pandas json_normalize

Apple M4 Pro (14 cores, 24 GB RAM, macOS), best-of-3 warm: ~3.6x faster than `pandas.json_normalize` on 800k orders.

In [ ]:
import time
import pandas as real_pd
import json

def chdb_flatten():
    d = pd.read_json("data/orders_large.json")
    flat = d.explode("items")
    flat_pdf = flat.to_pandas().dropna(subset=["items"])
    flat_pdf["country"] = flat_pdf["customer"].apply(lambda c: c["country"])
    flat_pdf["revenue"] = flat_pdf["items"].apply(lambda i: i["qty"] * i["price"])
    return flat_pdf.groupby("country")["revenue"].sum().round(2).sort_index()

def pandas_flatten():
    with open("data/orders_large.json") as f:
        records = json.load(f)
    flat_pdf = real_pd.json_normalize(records, record_path="items", meta=[["customer", "country"]])
    flat_pdf["revenue"] = flat_pdf["qty"] * flat_pdf["price"]
    return flat_pdf.groupby("customer.country")["revenue"].sum().round(2).sort_index()

def best_of_3(fn):
    fn()
    best = float("inf")
    for _ in range(3):
        t0 = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t0)
    return best

ds_s = best_of_3(chdb_flatten)
pd_s = best_of_3(pandas_flatten)
print(f"import pandas as pd (json_normalize):      {pd_s:.3f}s")
print(f"import chdb.datastore as pd (explode):     {ds_s:.3f}s")
print(f"speedup:                                   {pd_s / ds_s:.1f}x")